In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


**# Find the Top 10 Assets with Highest Energy Consumption**

In [9]:
# Top 10 Assets by Total Energy Consumption

top_10_assets_df = spark.sql("""
SELECT
    asset_id,
    ROUND(SUM(hourly_energy_consumption), 2) AS total_energy_consumption
FROM dbo.gold_fact_energy
GROUP BY asset_id
ORDER BY total_energy_consumption DESC
LIMIT 10
""")

# Validation
top_10_assets_df.show(10, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 14, Finished, Available, Finished, False)

+--------+------------------------+
|asset_id|total_energy_consumption|
+--------+------------------------+
|A041    |2500.94                 |
|A026    |2402.08                 |
|A039    |2335.29                 |
|A001    |2281.54                 |
|A006    |2263.97                 |
|A049    |2262.7                  |
|A034    |2239.42                 |
|A009    |2225.14                 |
|A021    |2217.28                 |
|A029    |2212.8                  |
+--------+------------------------+



**# 2. Calculate Average Daily Energy Consumption for Each Site**

In [8]:
avg_daily_energy_df = spark.sql("""
SELECT
    site_id,
    date_key,
    ROUND(AVG(hourly_energy_consumption), 2) AS avg_daily_energy_consumption
FROM dbo.gold_fact_energy
GROUP BY site_id, date_key
""")

# Validation
avg_daily_energy_df.show(20, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 13, Finished, Available, Finished, False)

+-------+----------+----------------------------+
|site_id|date_key  |avg_daily_energy_consumption|
+-------+----------+----------------------------+
|S002   |2026-08-02|21.94                       |
|S003   |2026-08-02|20.35                       |
|S005   |2026-08-02|21.72                       |
|S001   |2026-08-02|22.05                       |
|S004   |2026-08-02|20.51                       |
|S004   |2026-08-01|20.66                       |
|S003   |2026-08-01|21.42                       |
|S005   |2026-08-01|20.53                       |
|S002   |2026-08-01|20.69                       |
|S001   |2026-08-01|20.88                       |
|S002   |2026-08-03|19.81                       |
|S003   |2026-08-03|22.18                       |
|S005   |2026-08-03|21.21                       |
|S001   |2026-08-03|20.06                       |
|S004   |2026-08-03|21.5                        |
|S003   |2026-08-04|22.14                       |
|S001   |2026-08-04|23.21                       |


**# 3. Identify Assets that Generated More Than 10 Faults in Last 30 Days**

In [14]:
# Assets that Generated More Than 10 Faults in the Last 30 Days

assets_with_high_faults_df = spark.sql("""
SELECT
    asset_id,
    COUNT(*) AS fault_count
FROM dbo.gold_fact_event
WHERE UPPER(event_type) = 'FAULT'
  AND date_key >= DATE_SUB(CURRENT_DATE(), 30)
GROUP BY asset_id
HAVING COUNT(*) > 10
ORDER BY fault_count DESC
""")

# Validation
assets_with_high_faults_df.show(20, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 19, Finished, Available, Finished, False)

+--------+-----------+
|asset_id|fault_count|
+--------+-----------+
+--------+-----------+



# **4. Find assets that have not reported telemetry for the last 24 hours. **

In [15]:
# Assets with No Telemetry in the Last 24 Hours

inactive_assets_df = spark.sql("""
SELECT
    asset_id,
    MAX(date_key) AS last_reported_date
FROM dbo.gold_fact_energy
GROUP BY asset_id
HAVING MAX(date_key) < DATE_SUB(CURRENT_DATE(), 1)
ORDER BY last_reported_date
""")

# Validation
inactive_assets_df.show(20, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 20, Finished, Available, Finished, False)

+--------+------------------+
|asset_id|last_reported_date|
+--------+------------------+
|A043    |2026-08-04        |
|A017    |2026-08-04        |
|A008    |2026-08-04        |
|A003    |2026-08-04        |
|A023    |2026-08-04        |
|A014    |2026-08-04        |
|A029    |2026-08-04        |
|A004    |2026-08-04        |
|A032    |2026-08-04        |
|A045    |2026-08-04        |
|A006    |2026-08-04        |
|A050    |2026-08-04        |
|A049    |2026-08-04        |
|A038    |2026-08-04        |
|A039    |2026-08-04        |
|A007    |2026-08-04        |
|A010    |2026-08-04        |
|A016    |2026-08-04        |
|A018    |2026-08-04        |
|A011    |2026-08-04        |
+--------+------------------+
only showing top 20 rows



**## 5. Calculate Hourly Utilization for Each Building**

In [16]:
# Hourly Utilization by Building

building_hourly_utilization_df = spark.sql("""
SELECT
    building_id,
    date_key,
    hour,
    SUM(reading_count) AS total_readings
FROM dbo.gold_fact_energy
GROUP BY
    building_id,
    date_key,
    hour
ORDER BY
    building_id,
    date_key,
    hour
""")

# Validation
building_hourly_utilization_df.show(20, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 21, Finished, Available, Finished, False)

+-----------+----------+-------------------+--------------+
|building_id|date_key  |hour               |total_readings|
+-----------+----------+-------------------+--------------+
|B001       |2026-08-01|2026-08-01 00:00:00|2             |
|B001       |2026-08-01|2026-08-01 01:00:00|7             |
|B001       |2026-08-01|2026-08-01 02:00:00|9             |
|B001       |2026-08-01|2026-08-01 03:00:00|7             |
|B001       |2026-08-01|2026-08-01 04:00:00|10            |
|B001       |2026-08-01|2026-08-01 05:00:00|11            |
|B001       |2026-08-01|2026-08-01 06:00:00|5             |
|B001       |2026-08-01|2026-08-01 07:00:00|11            |
|B001       |2026-08-01|2026-08-01 08:00:00|7             |
|B001       |2026-08-01|2026-08-01 09:00:00|7             |
|B001       |2026-08-01|2026-08-01 10:00:00|7             |
|B001       |2026-08-01|2026-08-01 11:00:00|9             |
|B001       |2026-08-01|2026-08-01 12:00:00|8             |
|B001       |2026-08-01|2026-08-01 13:00

**# 6. Identify Sites with Abnormal Increases in Power Consumption**

In [17]:
# Sites with Abnormal Increase in Power Consumption

abnormal_power_consumption_df = spark.sql("""
WITH daily_power AS (
    SELECT
        site_id,
        date_key,
        AVG(avg_power_consumption) AS daily_avg_power
    FROM dbo.gold_fact_energy
    GROUP BY site_id, date_key
),
site_baseline AS (
    SELECT
        site_id,
        AVG(daily_avg_power) AS historical_avg_power
    FROM daily_power
    GROUP BY site_id
)
SELECT
    d.site_id,
    d.date_key,
    ROUND(d.daily_avg_power, 2) AS daily_avg_power,
    ROUND(s.historical_avg_power, 2) AS historical_avg_power
FROM daily_power d
INNER JOIN site_baseline s
    ON d.site_id = s.site_id
WHERE d.daily_avg_power > s.historical_avg_power * 1.5
ORDER BY d.daily_avg_power DESC
""")

# Validation
abnormal_power_consumption_df.show(20, truncate=False)

StatementMeta(, 51142721-c097-44de-8864-58ffe98b593d, 22, Finished, Available, Finished, False)

+-------+--------+---------------+--------------------+
|site_id|date_key|daily_avg_power|historical_avg_power|
+-------+--------+---------------+--------------------+
+-------+--------+---------------+--------------------+

